# recency: pooled direct logistic validation

One model per lead; all oblasts pooled, no identity encoding, calendar or STL predictors. Leads 1–6 are primary; lead 0 is separate; 12/24 are secondary. Each fold starts from scratch on TRAIN ending 25 hours before VALID. Scaling, rates and coefficients fit TRAIN only. C is selected by pooled VALID Brier over four independent folds; these are selection scores, not unbiased final performance. No resampling or TEST evaluation.


In [1]:
from pathlib import Path
import sys
import pandas as pd
root = Path.cwd() if Path("src/airalerts").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))
from airalerts.evaluate import latest_results
out = root / "reports/model"
latest = latest_results(out / "results.csv")


In [2]:
results = pd.read_csv(out / "recency_validation.csv", dtype={"fold_id": str})
summary = results.loc[results.fold_id.eq("pooled") & results.scope.eq("ALL")]
print(summary[["model", "lead", "n", "brier", "bss_vs_clim", "bss_vs_persistence", "bss_vs_ses", "bss_ci_low", "bss_ci_high", "ece"]].to_string(index=False))
print(pd.read_csv(out / "recency_selected.csv").to_string(index=False))


          model  lead     n    brier  bss_vs_clim  bss_vs_persistence  bss_vs_ses  bss_ci_low  bss_ci_high      ece
  logreg_A_hl14     0 35136 0.055548     0.430907            0.029876    0.340144    0.403669     0.459402 0.018985
  logreg_A_hl30     0 35136 0.055061     0.435894            0.038378    0.345927    0.408822     0.464395 0.015745
  logreg_A_hl60     0 35136 0.054951     0.437015            0.040289    0.347227    0.409878     0.465301 0.013849
 logreg_A_hlinf     0 35136 0.055035     0.436156            0.038825    0.346231    0.408764     0.464261 0.013652
 logreg_AB_hl14     1 35112 0.087459     0.104086            0.046883    0.049539    0.069380     0.142694 0.027087
 logreg_AB_hl30     1 35112 0.087519     0.103471            0.046230    0.048887    0.069061     0.142468 0.027698
 logreg_AB_hl60     1 35112 0.087649     0.102136            0.044810    0.047471    0.067913     0.141394 0.028171
logreg_AB_hlinf     1 35112 0.087916     0.099399            0.041897   

## Recency setup

Use A or AB per lead according to the context gain rule, fixing its previously selected C. Fit scaler and logistic loss with w=0.5**(age_days/half_life), age from exclusive TRAIN end. Select half-life 14/30/60/inf on pooled VALID; inf is the unweighted sensitivity.


In [3]:
print(pd.read_csv(out / "recency_selection.csv").to_string(index=False))


 lead          model    brier     C  half_life base_model  relative_brier_gain_vs_base    ci_low  ci_high           run_id
    0  logreg_A_hl60 0.054951  0.01       60.0   logreg_A                     0.001524 -0.000789 0.003807 c1c884d499a8a449
    1 logreg_AB_hl14 0.087459  0.01       14.0  logreg_AB                     0.005204  0.000543 0.009566 c1c884d499a8a449
    2 logreg_AB_hl14 0.090837  0.10       14.0  logreg_AB                     0.004769  0.000696 0.009222 c1c884d499a8a449
    3 logreg_AB_hl30 0.091721  0.01       30.0  logreg_AB                     0.003011  0.000751 0.005258 c1c884d499a8a449
    4 logreg_AB_hl30 0.092499  0.01       30.0  logreg_AB                     0.003913  0.001518 0.005857 c1c884d499a8a449
    5 logreg_AB_hl30 0.092985  0.01       30.0  logreg_AB                     0.003901  0.001566 0.006145 c1c884d499a8a449
    6 logreg_AB_hl14 0.093379  0.10       14.0  logreg_AB                     0.003833  0.000095 0.007722 c1c884d499a8a449
   12  logreg_A_

## Calibration

ECE uses ten bins. CIs pair all regions within circular seven-day UTC blocks; one-day blocks are sensitivity only. Flags are computed separately from each fold TRAIN target rate, below 3% or above 95%; per-oblast and pooled excluded-flagged metrics are in results.csv.


![Calibration: recency](../reports/model/recency_calibration.png)

Selected half-lives by lead are {0: 60.0, 1: 14.0, 2: 14.0, 3: 30.0, 4: 30.0, 5: 30.0, 6: 14.0, 12: 60.0, 24: inf}. The paired gain intervals versus each unweighted base are reported, including intervals crossing zero. These are validation selections only; no HGB, frozen final configuration or TEST evaluation is run.
